<a href="https://colab.research.google.com/github/dtkachepa/AI-and-Geometry-based-Dose-Prediction-for-Radiotherapy/blob/main/Copy_of_RANDose_Runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

REPO_URL = "https://github.com/iannyFARUE/Randose.git"
REPO_DIR = "/content/Randose"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Already cloned — discard any local changes and pull latest
    print("Repo exists — pulling latest changes...")
    !git -C {REPO_DIR} fetch --all
    !git -C {REPO_DIR} reset --hard origin/main
    !git -C {REPO_DIR} pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -5

Cloning repo...
Cloning into '/content/Randose'...
remote: Enumerating objects: 4277, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 4277 (delta 35), reused 45 (delta 20), pack-reused 4212 (from 2)
Receiving objects: 100% (4277/4277), 528.68 MiB | 56.39 MiB/s, done.
Resolving deltas: 100% (1524/1524), done.
Updating files: 100% (4036/4036), done.
/content/Randose
615dcda (HEAD -> main, origin/main, origin/HEAD) added two model comparisons
5792524 updated color map
bee47b0 fixed colormap
02955fc added boundary aware loss
33651be fix: added changes to notebook


In [ ]:
# Mount Google Drive for persistent checkpoint storage.
# Checkpoints saved here survive Colab disconnects and credit resets.
from google.colab import drive
drive.mount('/content/drive')

# Set your checkpoint directory on Drive (created automatically if it doesn't exist)
GDRIVE_CKPT_DIR = '/content/drive/MyDrive/RANDose_checkpoints'

import os
os.makedirs(GDRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint directory: {GDRIVE_CKPT_DIR}')
print('Existing checkpoints:', os.listdir(GDRIVE_CKPT_DIR) if os.path.exists(GDRIVE_CKPT_DIR) else 'none')

Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/RANDose_checkpoints
Existing checkpoints: []


In [ ]:
!pip install -U uv

In [ ]:
!uv pip install -e .

Using Python 3.12.12 environment at: /usr
Resolved 113 packages in 567ms
Prepared 22 packages in 1.46s
Uninstalled 8 packages in 190ms
Installed 22 packages in 43ms
 + argparse==1.4.0
 + bitsandbytes==0.49.2
 + colt5-attention==0.11.1
 - datasets==4.0.0
 + datasets==4.8.2
 + einops-exts==0.0.4
 + einx==0.4.2
 - huggingface-hub==1.7.1
 + huggingface-hub==0.36.2
 + hyper-connections==0.4.9
 - joblib==1.5.3
 + joblib==1.3.2
 + local-attention==1.11.2
 + loguru==0.7.3
 - matplotlib==3.10.0
 + matplotlib==3.10.8
 - numpy==2.0.2
 + numpy==2.4.3
 - pyarrow==18.1.0
 + pyarrow==23.0.1
 + pytorch-msssim==1.0.0
 + randose==0.1.0 (from file:///content/Randose)
 - scikit-learn==1.6.1
 + scikit-learn==1.5.2
 + simpleitk==2.5.3
 + torch-einops-utils==0.0.30
 - transformers==5.0.0
 + transformers==4.57.6
 + vector-quantize-pytorch==1.27.21
 + zetascale==2.8.8


In [ ]:
!uv run python utils/data.py

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Installed 110 packages in 168ms
['provided-data/train-pats/pt_62', 'provided-data/train-pats/pt_184', 'provided-data/train-pats/pt_94', 'provided-data/train-pats/pt_12', 'provided-data/train-pats/pt_127', 'provided-data/train-pats/pt_96', 'provided-data/train-pats/pt_5', 'provided-data/train-pats/pt_132', 'provided-data/train-pats/pt_130', 'provided-data/train-pats/pt_113', 'provided-data/train-pats/pt_195', 'provided-data/train-pats/pt_56', 'provided-data/train-pats/pt_50', 'provided-data/train-pats/pt_42', 'provided-data/train-pats/pt_147', 'provided-data/train-pats/pt_112', 'provided-data/train-pats/pt_153', 'provided-data/train-pats/pt_58', 'provided-data/train-pats/pt_167', 'provided-data/train-pats/pt_163', 'provided-data/train-pats/pt_124', 'provided-data/train-pats/pt_138', 'provided-data/train-pats/pt_49', 'provided-data/train-pats/pt_178', 'provided-data/train-pats/pt_135', 'provided

In [ ]:
import os

GDRIVE_CKPT_DIR = '/content/drive/MyDrive/RANDose_checkpoints'
LATEST_CKPT     = os.path.join(GDRIVE_CKPT_DIR, 'latest.pkl')

# Auto-detect whether to resume: if a checkpoint exists on Drive, resume from it.
if os.path.exists(LATEST_CKPT):
    resume_flag = f'--resume {LATEST_CKPT}'
    print(f'[RESUME] Found checkpoint at {LATEST_CKPT}. Resuming training.')
else:
    resume_flag = ''
    print('[FRESH START] No checkpoint found. Starting from scratch.')

train_cmd = f"""python train.py \\
    --model Model_RANDose_MambaA \\
    --loss Loss_DC_PTV \\
    --batch_size 2 \\
    --list_GPU_ids 0 \\
    --max_iter 80000 \\
    --project_name RANDose_reproduction \\
    --checkpoint_gdrive_dir {GDRIVE_CKPT_DIR} \\
    {resume_flag}"""

print('Running:', train_cmd)
get_ipython().system(train_cmd)

In [ ]:
# ─── EVALUATION ────────────────────────────────────────────────────────────────
# Runs inference on the 100 test patients (pt_241 → pt_340),
# saves predictions to RANDose_reproduction/Prediction/<patient_id>/dose.nii.gz,
# then computes Dose score and DVH score.
#
# Use best_val_evaluation_index.pkl for the best generalising checkpoint,
# or latest.pkl / best_train_loss.pkl if you prefer.

#--model_path RANDose_reproduction/best_val_evaluation_index.pkl \

BASELINE_CKPT = '/content/drive/MyDrive/"Dose Prediction for Radiotherapy"/RANDose_Baseline_checkpoints/best_val_evaluation_index.pkl'
!python test.py \
    --GPU_id 0 \
    --model Model_MTASP \
    --model_path {BASELINE_CKPT} \
    --project_name RANDose_reproduction \
    --TTA False

E0000 00:00:1773865565.108623   16466 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773865565.111891   16466 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773865565.120257   16466 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773865565.120280   16466 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773865565.120295   16466 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773865565.120296   16466 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
# ─── EVALUATION: RANDose + MambaA ──────────────────────────────────────────────
# Runs inference for the RANDose + MambaA model on the 100 test patients.
# Predictions are saved to RANDose_MambaA/Prediction/<patient_id>/dose.nii.gz
# (separate from the baseline RANDose_reproduction/ directory).
#
# Point --model_path at the best checkpoint from your MambaA training run.

MAMBA_A_CKPT = '/content/drive/MyDrive/"Dose Prediction for Radiotherapy"/RANDose_MambaA_checkpoints/best_val_evaluation_index.pkl'

!python test.py \
    --GPU_id 0 \
    --model Model_RANDose_MambaA \
    --model_path {MAMBA_A_CKPT} \
    --project_name RANDose_MambaA \
    --TTA False


E0000 00:00:1773867844.305904   27053 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773867844.309132   27053 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773867844.317487   27053 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773867844.317503   27053 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773867844.317505   27053 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773867844.317506   27053 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
# ─── EVALUATION: RANDose + MambaB ──────────────────────────────────────────────
# Runs inference for the RANDose + MambaB model on the 100 test patients.
# Predictions are saved to RANDose_MambaA/Prediction/<patient_id>/dose.nii.gz
# (separate from the baseline RANDose_reproduction/ directory).
#
# Point --model_path at the best checkpoint from your MambaA training run.

MAMBA_B_CKPT = '/content/drive/MyDrive/"Dose Prediction for Radiotherapy"/RANDose_MambaB_checkpoints/best_val_evaluation_index.pkl'

!python test.py \
    --GPU_id 0 \
    --model Model_RANDose_MambaB \
    --model_path {MAMBA_B_CKPT} \
    --project_name RANDose_MambaB \
    --TTA False


E0000 00:00:1773868034.628875   28138 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773868034.632108   28138 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773868034.640401   28138 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773868034.640414   28138 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773868034.640416   28138 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773868034.640417   28138 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
# Display the saved figures inline and copy them to Google Drive
from IPython.display import Image, display
import shutil, glob

SAVE_DIR    = 'RANDose_reproduction/VisualResults'
GDRIVE_VIS  = '/content/drive/MyDrive/RANDose_checkpoints/VisualResults'
os.makedirs(GDRIVE_VIS, exist_ok=True)

for fig_path in sorted(glob.glob(f'{SAVE_DIR}/*.png')):
    print(fig_path)
    display(Image(filename=fig_path))
    shutil.copy2(fig_path, GDRIVE_VIS)

print(f'Figures also saved to Google Drive: {GDRIVE_VIS}')

In [ ]:
"""
Visual dose distribution comparison  —  RANDose paper Fig 3 style.

Layout:
    Row 0: Ground Truth  (axial | coronal | sagittal)
    Row 1: Prediction    (axial | coronal | sagittal)
    Row 2: |Difference|  (axial | coronal | sagittal)  — jet, 0–70 Gy

All three rows share the same jet colormap and 0–70 Gy scale.
"""

import os
import numpy as np
import SimpleITK as sitk
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

# ── CONFIG ───────────────────────────────────────────────────────────────────
GT_DIR   = 'OpenKBP_C3D'
PRED_DIR = 'RANDose_reproduction/Prediction'
SAVE_DIR = 'RANDose_reproduction/VisualResults'
os.makedirs(SAVE_DIR, exist_ok=True)

PATIENT_ID  = 'pt_241'   # change to any test patient
DOSE_ALPHA  = 0.6
MAX_DOSE_GY = 70.
DOSE_CMAP   = 'jet'


def load_volume(path, dtype=None):
    img = sitk.ReadImage(path, dtype) if dtype else sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img).astype(np.float32)


def centre_slices(vol):
    z, h, w = vol.shape
    return z // 2, h // 2, w // 2


def get_slices(vol, az, co, sa):
    return vol[az, :, :], vol[:, co, :], vol[:, :, sa]


def overlay_ax(ax, ct_slice, dose_slice, mask_slice, title=''):
    """Draw CT (grey) with masked dose heatmap — shared jet 0–70 Gy scale."""
    ct_disp = np.clip(ct_slice, -1024, 1500)
    ct_disp = (ct_disp - ct_disp.min()) / (ct_disp.max() - ct_disp.min() + 1e-8)
    dose_masked = np.ma.masked_where(mask_slice < 1, dose_slice)
    ax.imshow(ct_disp, cmap='gray', origin='lower', aspect='equal')
    ax.imshow(dose_masked, cmap=DOSE_CMAP, alpha=DOSE_ALPHA,
              origin='lower', aspect='equal', vmin=0, vmax=MAX_DOSE_GY)
    ax.set_title(title, fontsize=8, pad=2)
    ax.axis('off')


# ── LOAD ─────────────────────────────────────────────────────────────────────
pred_dose_path = os.path.join(PRED_DIR, PATIENT_ID, 'dose.nii.gz')
if not os.path.exists(pred_dose_path):
    raise FileNotFoundError(f'No prediction for {PATIENT_ID} — run test.py first.')

ct      = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'CT.nii.gz'), sitk.sitkInt16)
gt_dose = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'dose.nii.gz'))
pr_dose = load_volume(pred_dose_path)
mask    = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'possible_dose_mask.nii.gz'), sitk.sitkUInt8)

az, co, sa = centre_slices(gt_dose)
planes     = ['Axial', 'Coronal', 'Sagittal']
row_labels = ['Ground Truth', 'Prediction', '|Difference|']

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(9, 8),
                         gridspec_kw={'hspace': 0.05, 'wspace': 0.05})
fig.suptitle(f'Dose Distribution — {PATIENT_ID}', fontsize=11, y=0.98)

for col, plane in enumerate(planes):
    ct_sl   = get_slices(ct,      az, co, sa)[col]
    gt_sl   = get_slices(gt_dose, az, co, sa)[col]
    pr_sl   = get_slices(pr_dose, az, co, sa)[col]
    mask_sl = get_slices(mask,    az, co, sa)[col]
    diff_sl = np.abs(gt_sl - pr_sl)

    overlay_ax(axes[0, col], ct_sl, gt_sl,   mask_sl, title=plane)
    overlay_ax(axes[1, col], ct_sl, pr_sl,   mask_sl)
    overlay_ax(axes[2, col], ct_sl, diff_sl, mask_sl)

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=9, labelpad=4)
    axes[row, 0].yaxis.set_visible(True)
    axes[row, 0].tick_params(left=False, labelleft=False)

# Single shared colorbar
norm    = Normalize(vmin=0, vmax=MAX_DOSE_GY)
cbar_ax = fig.add_axes([0.92, 0.06, 0.018, 0.82])
sm      = cm.ScalarMappable(cmap=DOSE_CMAP, norm=norm)
sm.set_array([])
cbar    = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Dose (Gy)', fontsize=8)

save_path = os.path.join(SAVE_DIR, f'{PATIENT_ID}_dose_comparison.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close(fig)
print(f'Saved: {save_path}')

In [ ]:
"""
Two-model dose comparison  —  RANDose paper Fig 3 style (extended).

Layout (5 rows × 3 planes):
    Row 0: Ground Truth                    — reference, jet 0–70 Gy
    Row 1: RANDose (Baseline) Prediction   — jet 0–70 Gy
    Row 2: RANDose |Difference|            — jet 0–diff_max Gy
    Row 3: RANDose + MambaA Prediction     — jet 0–70 Gy
    Row 4: RANDose + MambaA |Difference|   — jet 0–diff_max Gy
    Row 5: RANDose + MambaB Prediction     — jet 0–70 Gy
    Row 6: RANDose + MambaB |Difference|   — jet 0–diff_max Gy
"""

import os
import numpy as np
import SimpleITK as sitk
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

# ── CONFIG ───────────────────────────────────────────────────────────────────
GT_DIR          = 'OpenKBP_C3D'
PRED_DIR_BASE   = 'RANDose_reproduction/Prediction'
PRED_DIR_MAMBA  = 'RANDose_MambaA/Prediction'
SAVE_DIR        = 'RANDose_reproduction/VisualResults'
os.makedirs(SAVE_DIR, exist_ok=True)

PATIENT_ID  = 'pt_241'
DOSE_ALPHA  = 0.6
MAX_DOSE_GY = 70.
DOSE_CMAP   = 'jet'

# Row label text and a short description shown as a subtitle per row
ROW_META = [
    ('Ground Truth',            'Reference dose from treatment plan'),
    ('RANDose\n(Baseline)',     'Predicted dose — original RANDose model'),
    ('RANDose\n|Difference|',   'Absolute error vs ground truth'),
    ('RANDose + Mamba(sequential)\n','Predicted dose — with Mamba SSM integration'),
    ('RANDose + Mamba(sequential) \n|Difference|',    'Absolute error vs ground truth'),
    # ('RANDose + Mamba(parallel)\n','Predicted dose — with Mamba SSM integration'),
    # ('RANDose + Mamba(parallel) \n|Difference|',    'Absolute error vs ground truth'),
]


def load_volume(path, dtype=None):
    img = sitk.ReadImage(path, dtype) if dtype else sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img).astype(np.float32)

def centre_slices(vol):
    z, h, w = vol.shape
    return z // 2, h // 2, w // 2

def get_slices(vol, az, co, sa):
    return vol[az, :, :], vol[:, co, :], vol[:, :, sa]

def overlay_ax(ax, ct_slice, dose_slice, mask_slice, vmin, vmax, title=''):
    ct_disp = np.clip(ct_slice, -1024, 1500)
    ct_disp = (ct_disp - ct_disp.min()) / (ct_disp.max() - ct_disp.min() + 1e-8)
    dose_masked = np.ma.masked_where(mask_slice < 1, dose_slice)
    ax.imshow(ct_disp, cmap='gray', origin='lower', aspect='equal')
    ax.imshow(dose_masked, cmap=DOSE_CMAP, alpha=DOSE_ALPHA,
              origin='lower', aspect='equal', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=8, pad=2)
    ax.axis('off')


# ── LOAD ─────────────────────────────────────────────────────────────────────
for label, path in [('Baseline', os.path.join(PRED_DIR_BASE,  PATIENT_ID, 'dose.nii.gz')),
                    ('MambaA',   os.path.join(PRED_DIR_MAMBA, PATIENT_ID, 'dose.nii.gz'))]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'{label} prediction not found: {path} — run test.py first.')

ct       = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'CT.nii.gz'), sitk.sitkInt16)
gt_dose  = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'dose.nii.gz'))
pr_base  = load_volume(os.path.join(PRED_DIR_BASE,  PATIENT_ID, 'dose.nii.gz'))
pr_mamba = load_volume(os.path.join(PRED_DIR_MAMBA, PATIENT_ID, 'dose.nii.gz'))
mask     = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'possible_dose_mask.nii.gz'), sitk.sitkUInt8)

az, co, sa = centre_slices(gt_dose)
planes     = ['Axial', 'Coronal', 'Sagittal']

diff_base  = np.abs(gt_dose - pr_base)
diff_mamba = np.abs(gt_dose - pr_mamba)
mask_flat  = mask > 0
diff_max   = float(np.percentile(
    np.concatenate([diff_base[mask_flat], diff_mamba[mask_flat]]), 95
)) if mask_flat.any() else 20.

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(5, 3, figsize=(11, 14),
                         gridspec_kw={'hspace': 0.06, 'wspace': 0.04})
fig.suptitle(f'Dose Distribution Comparison — {PATIENT_ID}', fontsize=12, y=0.995)

for col, plane in enumerate(planes):
    ct_sl     = get_slices(ct,       az, co, sa)[col]
    gt_sl     = get_slices(gt_dose,  az, co, sa)[col]
    base_sl   = get_slices(pr_base,  az, co, sa)[col]
    mamba_sl  = get_slices(pr_mamba, az, co, sa)[col]
    mask_sl   = get_slices(mask,     az, co, sa)[col]
    diff_b_sl = np.abs(gt_sl - base_sl)
    diff_m_sl = np.abs(gt_sl - mamba_sl)

    overlay_ax(axes[0, col], ct_sl, gt_sl,     mask_sl, 0, MAX_DOSE_GY, title=plane)
    overlay_ax(axes[1, col], ct_sl, base_sl,   mask_sl, 0, MAX_DOSE_GY)
    overlay_ax(axes[2, col], ct_sl, diff_b_sl, mask_sl, 0, diff_max)
    overlay_ax(axes[3, col], ct_sl, mamba_sl,  mask_sl, 0, MAX_DOSE_GY)
    overlay_ax(axes[4, col], ct_sl, diff_m_sl, mask_sl, 0, diff_max)

# ── Row labels (fig.text so axis('off') doesn't hide them) ───────────────────
# Get bounding boxes of the leftmost axes column to place labels precisely.
fig.canvas.draw()          # needed so get_position() is finalised
renderer = fig.canvas.get_renderer()

for row, (label, desc) in enumerate(ROW_META):
    ax = axes[row, 0]
    pos = ax.get_position()          # axes coords (0–1 in figure space)
    y_centre = pos.y0 + pos.height / 2

    # Bold row name
    fig.text(0.01, y_centre, label,
             ha='left', va='center', fontsize=8, fontweight='bold',
             multialignment='center')
    # Italic description below
    fig.text(0.01, y_centre - pos.height * 0.28, desc,
             ha='left', va='center', fontsize=6, style='italic', color='#444444',
             multialignment='center')

# Divider between the two model blocks (rows 2 → 3)
div_y = (axes[2, 0].get_position().y0 + axes[3, 0].get_position().y1) / 2
fig.add_artist(plt.Line2D([0.01, 0.91], [div_y, div_y],
                           transform=fig.transFigure,
                           color='#888888', linewidth=0.8, linestyle='--'))

# ── Colorbars ─────────────────────────────────────────────────────────────────
cbar_ax1 = fig.add_axes([0.93, 0.42, 0.018, 0.54])
sm1 = cm.ScalarMappable(cmap=DOSE_CMAP, norm=Normalize(vmin=0, vmax=MAX_DOSE_GY))
sm1.set_array([])
cb1 = fig.colorbar(sm1, cax=cbar_ax1)
cb1.set_label('Dose (Gy)', fontsize=8)

cbar_ax2 = fig.add_axes([0.93, 0.04, 0.018, 0.35])
sm2 = cm.ScalarMappable(cmap=DOSE_CMAP, norm=Normalize(vmin=0, vmax=diff_max))
sm2.set_array([])
cb2 = fig.colorbar(sm2, cax=cbar_ax2)
cb2.set_label('|Diff| (Gy)', fontsize=8)

save_path = os.path.join(SAVE_DIR, f'{PATIENT_ID}_model_comparison.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close(fig)
print(f'Saved: {save_path}')


Saved: RANDose_reproduction/VisualResults/pt_241_model_comparison.png


In [ ]:
"""
Three-model dose comparison — RANDose paper Fig 3 style (extended).

Layout (7 rows × 3 planes):
    Row 0: Ground Truth
    Row 1: RANDose (Baseline) Prediction
    Row 2: RANDose |Difference|
    Row 3: RANDose + MambaA Prediction
    Row 4: RANDose + MambaA |Difference|
    Row 5: RANDose + MambaB Prediction
    Row 6: RANDose + MambaB |Difference|
"""

import os
import numpy as np
import SimpleITK as sitk
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

# ── CONFIG ───────────────────────────────────────────────────────────────────
GT_DIR            = 'OpenKBP_C3D'
PRED_DIR_BASE     = 'RANDose_reproduction/Prediction'
PRED_DIR_MAMBA_A  = 'RANDose_MambaA/Prediction'
PRED_DIR_MAMBA_B  = 'RANDose_MambaB/Prediction'
SAVE_DIR          = 'RANDose_reproduction/VisualResults'
os.makedirs(SAVE_DIR, exist_ok=True)

PATIENT_ID  = 'pt_241'
DOSE_ALPHA  = 0.6
MAX_DOSE_GY = 70.
DOSE_CMAP   = 'jet'

ROW_META = [
    ('Ground Truth', 'Reference dose from treatment plan'),
    ('RANDose\n(Baseline)', 'Predicted dose — original RANDose model'),
    ('RANDose\n|Difference|', 'Absolute error vs ground truth'),
    ('RANDose + MambaA', 'Predicted dose — with MambaA integration'),
    ('RANDose + MambaA\n|Difference|', 'Absolute error vs ground truth'),
    ('RANDose + MambaB', 'Predicted dose — with MambaB integration'),
    ('RANDose + MambaB\n|Difference|', 'Absolute error vs ground truth'),
]

def load_volume(path, dtype=None):
    img = sitk.ReadImage(path, dtype) if dtype else sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img).astype(np.float32)

def centre_slices(vol):
    z, h, w = vol.shape
    return z // 2, h // 2, w // 2

def get_slices(vol, az, co, sa):
    return vol[az, :, :], vol[:, co, :], vol[:, :, sa]

def overlay_ax(ax, ct_slice, dose_slice, mask_slice, vmin, vmax, title=''):
    ct_disp = np.clip(ct_slice, -1024, 1500)
    ct_disp = (ct_disp - ct_disp.min()) / (ct_disp.max() - ct_disp.min() + 1e-8)
    dose_masked = np.ma.masked_where(mask_slice < 1, dose_slice)

    ax.imshow(ct_disp, cmap='gray', origin='lower', aspect='equal')
    ax.imshow(
        dose_masked,
        cmap=DOSE_CMAP,
        alpha=DOSE_ALPHA,
        origin='lower',
        aspect='equal',
        vmin=vmin,
        vmax=vmax
    )
    ax.set_title(title, fontsize=8, pad=2)
    ax.axis('off')

# ── CHECK FILES ──────────────────────────────────────────────────────────────
for label, path in [
    ('Baseline', os.path.join(PRED_DIR_BASE, PATIENT_ID, 'dose.nii.gz')),
    ('MambaA',   os.path.join(PRED_DIR_MAMBA_A, PATIENT_ID, 'dose.nii.gz')),
    ('MambaB',   os.path.join(PRED_DIR_MAMBA_B, PATIENT_ID, 'dose.nii.gz')),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'{label} prediction not found: {path} — run test.py first.')

# ── LOAD ─────────────────────────────────────────────────────────────────────
ct         = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'CT.nii.gz'), sitk.sitkInt16)
gt_dose    = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'dose.nii.gz'))
pr_base    = load_volume(os.path.join(PRED_DIR_BASE, PATIENT_ID, 'dose.nii.gz'))
pr_mamba_a = load_volume(os.path.join(PRED_DIR_MAMBA_A, PATIENT_ID, 'dose.nii.gz'))
pr_mamba_b = load_volume(os.path.join(PRED_DIR_MAMBA_B, PATIENT_ID, 'dose.nii.gz'))
mask       = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'possible_dose_mask.nii.gz'), sitk.sitkUInt8)

az, co, sa = centre_slices(gt_dose)
planes = ['Axial', 'Coronal', 'Sagittal']

diff_base    = np.abs(gt_dose - pr_base)
diff_mamba_a = np.abs(gt_dose - pr_mamba_a)
diff_mamba_b = np.abs(gt_dose - pr_mamba_b)

mask_flat = mask > 0
diff_max = float(np.percentile(
    np.concatenate([
        diff_base[mask_flat],
        diff_mamba_a[mask_flat],
        diff_mamba_b[mask_flat]
    ]),
    95
)) if mask_flat.any() else 20.

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(7, 3, figsize=(11, 19),
                         gridspec_kw={'hspace': 0.06, 'wspace': 0.04})
fig.suptitle(f'Dose Distribution Comparison — {PATIENT_ID}', fontsize=12, y=0.995)

for col, plane in enumerate(planes):
    ct_sl       = get_slices(ct, az, co, sa)[col]
    gt_sl       = get_slices(gt_dose, az, co, sa)[col]
    base_sl     = get_slices(pr_base, az, co, sa)[col]
    mamba_a_sl  = get_slices(pr_mamba_a, az, co, sa)[col]
    mamba_b_sl  = get_slices(pr_mamba_b, az, co, sa)[col]
    mask_sl     = get_slices(mask, az, co, sa)[col]

    diff_b_sl   = np.abs(gt_sl - base_sl)
    diff_a_sl   = np.abs(gt_sl - mamba_a_sl)
    diff_bb_sl  = np.abs(gt_sl - mamba_b_sl)

    overlay_ax(axes[0, col], ct_sl, gt_sl,      mask_sl, 0, MAX_DOSE_GY, title=plane)
    overlay_ax(axes[1, col], ct_sl, base_sl,    mask_sl, 0, MAX_DOSE_GY)
    overlay_ax(axes[2, col], ct_sl, diff_b_sl,  mask_sl, 0, diff_max)
    overlay_ax(axes[3, col], ct_sl, mamba_a_sl, mask_sl, 0, MAX_DOSE_GY)
    overlay_ax(axes[4, col], ct_sl, diff_a_sl,  mask_sl, 0, diff_max)
    overlay_ax(axes[5, col], ct_sl, mamba_b_sl, mask_sl, 0, MAX_DOSE_GY)
    overlay_ax(axes[6, col], ct_sl, diff_bb_sl, mask_sl, 0, diff_max)

# ── Row labels ────────────────────────────────────────────────────────────────
fig.canvas.draw()

for row, (label, desc) in enumerate(ROW_META):
    ax = axes[row, 0]
    pos = ax.get_position()
    y_centre = pos.y0 + pos.height / 2

    fig.text(0.01, y_centre, label,
             ha='left', va='center', fontsize=8, fontweight='bold',
             multialignment='center')

    fig.text(0.01, y_centre - pos.height * 0.28, desc,
             ha='left', va='center', fontsize=6, style='italic',
             color='#444444', multialignment='center')

# Optional divider lines between blocks
div_y_1 = (axes[2, 0].get_position().y0 + axes[3, 0].get_position().y1) / 2
div_y_2 = (axes[4, 0].get_position().y0 + axes[5, 0].get_position().y1) / 2

fig.add_artist(plt.Line2D([0.01, 0.91], [div_y_1, div_y_1],
                          transform=fig.transFigure,
                          color='#888888', linewidth=0.8, linestyle='--'))

fig.add_artist(plt.Line2D([0.01, 0.91], [div_y_2, div_y_2],
                          transform=fig.transFigure,
                          color='#888888', linewidth=0.8, linestyle='--'))

# ── Colorbars ─────────────────────────────────────────────────────────────────
cbar_ax1 = fig.add_axes([0.93, 0.50, 0.018, 0.43])
sm1 = cm.ScalarMappable(cmap=DOSE_CMAP, norm=Normalize(vmin=0, vmax=MAX_DOSE_GY))
sm1.set_array([])
cb1 = fig.colorbar(sm1, cax=cbar_ax1)
cb1.set_label('Dose (Gy)', fontsize=8)

cbar_ax2 = fig.add_axes([0.93, 0.08, 0.018, 0.35])
sm2 = cm.ScalarMappable(cmap=DOSE_CMAP, norm=Normalize(vmin=0, vmax=diff_max))
sm2.set_array([])
cb2 = fig.colorbar(sm2, cax=cbar_ax2)
cb2.set_label('|Diff| (Gy)', fontsize=8)

save_path = os.path.join(SAVE_DIR, f'{PATIENT_ID}_model_comparison.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close(fig)

print(f'Saved: {save_path}')

In [ ]:
"""
Three-model dose comparison — RANDose paper Fig 3 style (extended).

Layout (7 rows × 3 planes):
    Row 0: Ground Truth
    Row 1: RANDose (Baseline) Prediction
    Row 2: RANDose |Difference|
    Row 3: RANDose + Mamba(sequential) Prediction
    Row 4: RANDose + Mamba(sequential) |Difference|
    Row 5: RANDose + Mamba(parallel) Prediction
    Row 6: RANDose + Mamba(parallel) |Difference|
"""

import os
import numpy as np
import SimpleITK as sitk
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

# ── CONFIG ───────────────────────────────────────────────────────────────────
GT_DIR            = 'OpenKBP_C3D'
PRED_DIR_BASE     = 'RANDose_reproduction/Prediction'
PRED_DIR_MAMBA_A  = 'RANDose_MambaA/Prediction'
PRED_DIR_MAMBA_B  = 'RANDose_MambaB/Prediction'
SAVE_DIR          = 'images'
os.makedirs(SAVE_DIR, exist_ok=True)

PATIENT_ID  = 'pt_241'
DOSE_ALPHA  = 0.6
MAX_DOSE_GY = 70.
DOSE_CMAP   = 'jet'

ROW_META = [
    ('Ground Truth', 'Reference dose from treatment plan'),
    ('RANDose\n(Baseline)', 'Predicted dose — original RANDose model'),
    ('RANDose\n|Difference|', 'Absolute error vs ground truth'),
    ('RANDose + Mamba(sequential)', 'Predicted dose — with Sequential Mamba integration'),
    ('RANDose + Mamba(sequential)\n|Difference|', 'Absolute error vs ground truth'),
    ('RANDose + Mamba(parallel)', 'Predicted dose — with Parallel Mamba integration'),
    ('RANDose + Mamba(parallel)\n|Difference|', 'Absolute error vs ground truth'),
]

def load_volume(path, dtype=None):
    img = sitk.ReadImage(path, dtype) if dtype else sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img).astype(np.float32)

def centre_slices(vol):
    z, h, w = vol.shape
    return z // 2, h // 2, w // 2

def get_slices(vol, az, co, sa):
    return vol[az, :, :], vol[:, co, :], vol[:, :, sa]

def overlay_ax(ax, ct_slice, dose_slice, mask_slice, vmin, vmax, title=''):
    ct_disp = np.clip(ct_slice, -1024, 1500)
    ct_disp = (ct_disp - ct_disp.min()) / (ct_disp.max() - ct_disp.min() + 1e-8)
    dose_masked = np.ma.masked_where(mask_slice < 1, dose_slice)

    ax.imshow(ct_disp, cmap='gray', origin='lower', aspect='equal')
    ax.imshow(
        dose_masked,
        cmap=DOSE_CMAP,
        alpha=DOSE_ALPHA,
        origin='lower',
        aspect='equal',
        vmin=vmin,
        vmax=vmax
    )
    ax.set_title(title, fontsize=8, pad=2)
    ax.axis('off')

# ── CHECK FILES ──────────────────────────────────────────────────────────────
for label, path in [
    ('Baseline', os.path.join(PRED_DIR_BASE, PATIENT_ID, 'dose.nii.gz')),
    ('MambaA',   os.path.join(PRED_DIR_MAMBA_A, PATIENT_ID, 'dose.nii.gz')),
    ('MambaB',   os.path.join(PRED_DIR_MAMBA_B, PATIENT_ID, 'dose.nii.gz')),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'{label} prediction not found: {path} — run test.py first.')

# ── LOAD ─────────────────────────────────────────────────────────────────────
ct         = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'CT.nii.gz'), sitk.sitkInt16)
gt_dose    = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'dose.nii.gz'))
pr_base    = load_volume(os.path.join(PRED_DIR_BASE, PATIENT_ID, 'dose.nii.gz'))
pr_mamba_a = load_volume(os.path.join(PRED_DIR_MAMBA_A, PATIENT_ID, 'dose.nii.gz'))
pr_mamba_b = load_volume(os.path.join(PRED_DIR_MAMBA_B, PATIENT_ID, 'dose.nii.gz'))
mask       = load_volume(os.path.join(GT_DIR, PATIENT_ID, 'possible_dose_mask.nii.gz'), sitk.sitkUInt8)

az, co, sa = centre_slices(gt_dose)
planes = ['Axial', 'Coronal', 'Sagittal']

diff_base    = np.abs(gt_dose - pr_base)
diff_mamba_a = np.abs(gt_dose - pr_mamba_a)
diff_mamba_b = np.abs(gt_dose - pr_mamba_b)

mask_flat = mask > 0
diff_max = float(np.percentile(
    np.concatenate([
        diff_base[mask_flat],
        diff_mamba_a[mask_flat],
        diff_mamba_b[mask_flat]
    ]),
    95
)) if mask_flat.any() else 20.

# ── PLOT ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(7, 3, figsize=(15, 19),
                         gridspec_kw={'hspace': 0.06, 'wspace': 0.04})
fig.suptitle(f'Dose Distribution Comparison — {PATIENT_ID}', fontsize=12, y=0.995)

for col, plane in enumerate(planes):
    ct_sl       = get_slices(ct, az, co, sa)[col]
    gt_sl       = get_slices(gt_dose, az, co, sa)[col]
    base_sl     = get_slices(pr_base, az, co, sa)[col]
    mamba_a_sl  = get_slices(pr_mamba_a, az, co, sa)[col]
    mamba_b_sl  = get_slices(pr_mamba_b, az, co, sa)[col]
    mask_sl     = get_slices(mask, az, co, sa)[col]

    diff_b_sl   = np.abs(gt_sl - base_sl)
    diff_a_sl   = np.abs(gt_sl - mamba_a_sl)
    diff_bb_sl  = np.abs(gt_sl - mamba_b_sl)

    overlay_ax(axes[0, col], ct_sl, gt_sl,      mask_sl, 0, MAX_DOSE_GY, title=plane)
    overlay_ax(axes[1, col], ct_sl, base_sl,    mask_sl, 0, MAX_DOSE_GY)
    overlay_ax(axes[2, col], ct_sl, diff_b_sl,  mask_sl, 0, diff_max)
    overlay_ax(axes[3, col], ct_sl, mamba_a_sl, mask_sl, 0, MAX_DOSE_GY)
    overlay_ax(axes[4, col], ct_sl, diff_a_sl,  mask_sl, 0, diff_max)
    overlay_ax(axes[5, col], ct_sl, mamba_b_sl, mask_sl, 0, MAX_DOSE_GY)
    overlay_ax(axes[6, col], ct_sl, diff_bb_sl, mask_sl, 0, diff_max)

# ── Row labels ────────────────────────────────────────────────────────────────
fig.canvas.draw()

for row, (label, desc) in enumerate(ROW_META):
    ax = axes[row, 0]
    pos = ax.get_position()
    y_centre = pos.y0 + pos.height / 2

    fig.text(0.01, y_centre, label,
             ha='left', va='center', fontsize=8, fontweight='bold',
             multialignment='center')

    fig.text(0.01, y_centre - pos.height * 0.28, desc,
             ha='left', va='center', fontsize=6, style='italic',
             color='#444444', multialignment='center')

# Optional divider lines between blocks
div_y_0 = (axes[0, 0].get_position().y0 + axes[1, 0].get_position().y1) / 2
div_y_1 = (axes[2, 0].get_position().y0 + axes[3, 0].get_position().y1) / 2
div_y_2 = (axes[4, 0].get_position().y0 + axes[5, 0].get_position().y1) / 2

fig.add_artist(plt.Line2D([0.01, 0.91], [div_y_0, div_y_0],
                          transform=fig.transFigure,
                          color='#888888', linewidth=0.8, linestyle='--'))

fig.add_artist(plt.Line2D([0.01, 0.91], [div_y_1, div_y_1],
                          transform=fig.transFigure,
                          color='#888888', linewidth=0.8, linestyle='--'))

fig.add_artist(plt.Line2D([0.01, 0.91], [div_y_2, div_y_2],
                          transform=fig.transFigure,
                          color='#888888', linewidth=0.8, linestyle='--'))

# ── Colorbars ─────────────────────────────────────────────────────────────────
cbar_ax1 = fig.add_axes([0.93, 0.50, 0.018, 0.43])
sm1 = cm.ScalarMappable(cmap=DOSE_CMAP, norm=Normalize(vmin=0, vmax=MAX_DOSE_GY))
sm1.set_array([])
cb1 = fig.colorbar(sm1, cax=cbar_ax1)
cb1.set_label('Dose (Gy)', fontsize=8)

cbar_ax2 = fig.add_axes([0.93, 0.08, 0.018, 0.35])
sm2 = cm.ScalarMappable(cmap=DOSE_CMAP, norm=Normalize(vmin=0, vmax=diff_max))
sm2.set_array([])
cb2 = fig.colorbar(sm2, cax=cbar_ax2)
cb2.set_label('|Diff| (Gy)', fontsize=8)

save_path = os.path.join(SAVE_DIR, f'{PATIENT_ID}_3_model_comparison.png')
fig.savefig(save_path, dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.close(fig)

print(f'Saved: {save_path}')

Saved: images/pt_241_3_model_comparison.png
